In [ ]:
# >>> Code commenté en français <<<
# Description :
# Cette cellule exécute les opérations suivantes :
# Projet : Rectangularisation des signaux d'altitude


<!-- Section Markdown : ## Étape 1 : Chargement des données... -->
## Étape 1 : Chargement des données

In [ ]:
# >>> Code commenté en français <<<
# Description :
# Cette cellule exécute les opérations suivantes :
# Importation du module : import h5py
import h5py
# Importation du module : import numpy as np
import numpy as np
# Importation du module : import pandas as pd
import pandas as pd
# Importation du module : import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from sklearn.metrics import mean_squared_error

# Définition d'une fonction : def load_altitude_from_record(h5_path, key):
def load_altitude_from_record(h5_path, key):
    with h5py.File(h5_path, 'r') as f:
# Traitement de données : group = f[key]
        group = f[key]
# Traitement de données : cols = [c.decode() for c in group['block0_items'][:]]
        cols = [c.decode() for c in group['block0_items'][:]]
# Traitement de données : values = group['block0_values'][:]
        values = group['block0_values'][:]
# Traitement de données : df = pd.DataFrame(values, columns=cols)
        df = pd.DataFrame(values, columns=cols)
    return df['ALT[m]'].values


<!-- Section Markdown : ## Étape 2 : Normalisation et métriques de synchronisation... -->
## Étape 2 : Normalisation et métriques de synchronisation

In [ ]:
# >>> Code commenté en français <<<
# Description :
# Cette cellule exécute les opérations suivantes :
# Définition d'une fonction : def normalize_signal(signal, target_length=200):
def normalize_signal(signal, target_length=200):
# Traitement de données : x_original = np.linspace(0, 1, len(signal))
    x_original = np.linspace(0, 1, len(signal))
# Traitement de données : f = interp1d(x_original, signal, kind='linear')
    f = interp1d(x_original, signal, kind='linear')
# Traitement de données : x_new = np.linspace(0, 1, target_length)
    x_new = np.linspace(0, 1, target_length)
    return f(x_new)

# Définition d'une fonction : def compute_mad_to_median(signals_matrix):
def compute_mad_to_median(signals_matrix):
# Traitement de données : median_curve = np.median(signals_matrix, axis=0)
    median_curve = np.median(signals_matrix, axis=0)
# Traitement de données : mad = np.mean(np.abs(signals_matrix - median_curve), axis=1)
    mad = np.mean(np.abs(signals_matrix - median_curve), axis=1)
    return np.mean(mad)

# Définition d'une fonction : def compute_envelope_width(signals_matrix):
def compute_envelope_width(signals_matrix):
# Traitement de données : return np.mean(np.max(signals_matrix, axis=0) - np.min(signals_matrix, axis=0))
    return np.mean(np.max(signals_matrix, axis=0) - np.min(signals_matrix, axis=0))


<!-- Section Markdown : ## Étape 3 : Rectangularisation simple... -->
## Étape 3 : Rectangularisation simple

In [ ]:
# >>> Code commenté en français <<<
# Description :
# Cette cellule exécute les opérations suivantes :
# Définition d'une fonction : def rectangularize_simple(signal):
def rectangularize_simple(signal):
# Traitement de données : x = np.linspace(0, 1, len(signal))
    x = np.linspace(0, 1, len(signal))
# Traitement de données : f = interp1d([x[0], x[-1]], [signal[0], signal[-1]])
    f = interp1d([x[0], x[-1]], [signal[0], signal[-1]])
    return f(x)


<!-- Section Markdown : ## Étape 4 : Visualisation des signaux... -->
## Étape 4 : Visualisation des signaux

In [ ]:
# >>> Code commenté en français <<<
# Description :
# Cette cellule exécute les opérations suivantes :
# Définition d'une fonction : def plot_superposed_signals(S, title="Signaux normalisés", show_median=True):
def plot_superposed_signals(S, title="Signaux normalisés", show_median=True):
# Traitement de données : plt.figure(figsize=(12, 5))
    plt.figure(figsize=(12, 5))
    for s in S:
# Traitement de données : plt.plot(s, color='gray', alpha=0.3)
        plt.plot(s, color='gray', alpha=0.3)
    if show_median:
# Traitement de données : median_curve = np.median(S, axis=0)
        median_curve = np.median(S, axis=0)
# Traitement de données : plt.plot(median_curve, color='red', linewidth=2, label='Médiane')
        plt.plot(median_curve, color='red', linewidth=2, label='Médiane')
    plt.title(title)
    plt.xlabel("Temps normalisé")
    plt.ylabel("Altitude (m)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


<!-- Section Markdown : ## Étape 5 : Pipeline d'exécution... -->
## Étape 5 : Pipeline d'exécution

In [ ]:
# >>> Code commenté en français <<<
# Description :
# Cette cellule exécute les opérations suivantes :
# Traitement de données : h5_path = "AFL1EB_cleaned_final.h5"
h5_path = "AFL1EB_cleaned_final.h5"
# Traitement de données : record_keys = [f"record_{i:02d}" for i in range(5)]
record_keys = [f"record_{i:02d}" for i in range(5)]

# Traitement de données : list_of_signals = [load_altitude_from_record(h5_path, key) for key in record_keys]
list_of_signals = [load_altitude_from_record(h5_path, key) for key in record_keys]
# Traitement de données : S = np.array([normalize_signal(sig) for sig in list_of_signals])
S = np.array([normalize_signal(sig) for sig in list_of_signals])
# Traitement de données : list_of_signals_rect = [rectangularize_simple(sig) for sig in list_of_signals]
list_of_signals_rect = [rectangularize_simple(sig) for sig in list_of_signals]
# Traitement de données : S_rect = np.array([normalize_signal(sig) for sig in list_of_signals_rect])
S_rect = np.array([normalize_signal(sig) for sig in list_of_signals_rect])

# Traitement de données : mad_sync = compute_mad_to_median(S)
mad_sync = compute_mad_to_median(S)
# Traitement de données : env_sync = compute_envelope_width(S)
env_sync = compute_envelope_width(S)
# Traitement de données : mad_sync_rect = compute_mad_to_median(S_rect)
mad_sync_rect = compute_mad_to_median(S_rect)
# Traitement de données : env_sync_rect = compute_envelope_width(S_rect)
env_sync_rect = compute_envelope_width(S_rect)

print("————————————————————————————————————————")
print(f"Synchronisation ORIGINALE (MAD à la médiane) : {mad_sync:.2f} m")
print(f"Synchronisation ORIGINALE (enveloppe moyenne) : {env_sync:.2f} m")
print()
print(f"Synchronisation RECTANGULARISÉE (MAD à la médiane) : {mad_sync_rect:.2f} m")
print(f"Synchronisation RECTANGULARISÉE (enveloppe moyenne) : {env_sync_rect:.2f} m")
print("————————————————————————————————————————")

# Traitement de données : plot_superposed_signals(S, title="Signaux normalisés — Originaux")
plot_superposed_signals(S, title="Signaux normalisés — Originaux")
# Traitement de données : plot_superposed_signals(S_rect, title="Signaux normalisés — Rectangularisés")
plot_superposed_signals(S_rect, title="Signaux normalisés — Rectangularisés")
